# ODI to Databricks Migration: WC_BADGE_D

**Conversion Timestamp:** 2024-07-30 12:00:00

This notebook migrates the `WC_BADGE_D` ODI process to Databricks Spark SQL. It handles the incremental loading of badge details into the `WC_BADGE_DETAILS_D` target table.

In [ ]:
# COMMAND ----------
dbutils.widgets.text("ETL_PROC_WID", "99999", "ETL Process Widget ID")
dbutils.widgets.text("DATASOURCE_NUM_ID", "1", "Datasource Number ID")
dbutils.widgets.text("ODI_SESS_NO", "-1", "ODI Session Number")

## ETL Parameters

In [ ]:
# COMMAND ----------
%sql
CREATE OR REPLACE TEMPORARY VIEW v_etl_parameters AS
SELECT
  CAST('${ETL_PROC_WID}' AS BIGINT) AS etl_proc_wid,
  CAST('${DATASOURCE_NUM_ID}' AS BIGINT) AS datasource_num_id,
  CAST('${ODI_SESS_NO}' AS BIGINT) AS odi_sess_no;

In [ ]:
# COMMAND ----------
display(spark.sql("SELECT *
FROM v_etl_parameters"))

## Staging Table: c_wc_badge_d

In [ ]:
# COMMAND ----------
-- SCEN_TASK_NO 30:
Drop staging
DROP TABLE IF EXISTS workspace.wc_badge_stg.c_wc_badge_d;

In [ ]:
# COMMAND ----------
-- SCEN_TASK_NO 40: Create staging
CREATE TABLE workspace.wc_badge_stg.c_wc_badge_d
(
    integration_id      STRING,
    badge_id            STRING,
    badge_status        BIGINT,
    contact_email       STRING,
    org_name            STRING,
    created_date        TIMESTAMP,
    last_updated_date   TIMESTAMP,
    etl_proc_wid        BIGINT
) USING DELTA;

In [ ]:
# COMMAND ----------
-- SCEN_TASK_NO 50: Load staging with incremental logic
INSERT INTO workspace.wc_badge_stg.c_wc_badge_d
SELECT 
    b.integration_id,
    b.badge_id,
    b.status,
    b.contact_email,
    b.organisation_name,
    b.creation_date,
    b.last_update_date,
    p.etl_proc_wid
FROM workspace.wc_source.src_badge_table AS b
CROSS JOIN v_etl_parameters AS p
WHERE b.last_update_date > to_date('2024-01-01', 'yyyy-MM-dd')
  AND b.last_update_date <= current_timestamp();

In [ ]:
# COMMAND ----------
%sql
SELECT COUNT(*) AS record_count FROM workspace.wc_badge_stg.c_wc_badge_d;

## Merge into Target: wc_badge_details_d

In [ ]:
# COMMAND ----------
-- SCEN_TASK_NO 100: Merge into target
MERGE INTO workspace.wc_badge.wc_badge_details_d AS T
USING (
    SELECT 
        integration_id,
        badge_id,
        badge_status        AS status,
        contact_email,
        org_name,
        created_date,
        last_updated_date
    FROM workspace.wc_badge_stg.c_wc_badge_d
) AS S
ON (T.integration_id = S.integration_id)
WHEN MATCHED THEN
    UPDATE SET
        T.status            = S.status,
        T.contact_email     = S.contact_email,
        T.org_name          = S.org_name,
        T.w_update_dt       = current_timestamp()
WHEN NOT MATCHED THEN
    INSERT (
        badge_id,
        status,
        contact_email,
        org_name,
        integration_id,
        w_insert_dt,
        w_update_dt
    ) VALUES (
        S.badge_id,
        S.status,
        S.contact_email,
        S.org_name,
        S.integration_id,
        current_timestamp(),
        current_timestamp()
    );

## Cleanup

In [ ]:
# COMMAND ----------
DROP TABLE IF EXISTS workspace.wc_badge_stg.c_wc_badge_d;

## Validation

In [ ]:
# COMMAND ----------
%sql
SELECT COUNT(*) AS final_target_record_count
FROM workspace.wc_badge.wc_badge_details_d;

In [ ]:
# COMMAND ----------
%sql
SELECT * FROM workspace.wc_badge.wc_badge_details_d
ORDER BY w_update_dt DESC
LIMIT 10;

## Conversion Notes and Manual Actions Required

1.  **Schema and Table Names**: Inferred schema names are `workspace.wc_badge_stg`, `workspace.wc_source`, and `workspace.wc_badge`. Please adjust these to your specific Databricks environment's schema definitions.
2.  **Datasource `src_badge_table` DDL**: The DDL for `workspace.wc_source.src_badge_table` is assumed to exist. Ensure it is created with appropriate Spark SQL data types and `USING DELTA` clause, matching the source structure.
3.  **Target Table `wc_badge_details_d` DDL**: The DDL for `workspace.wc_badge.wc_badge_details_d` is assumed to exist. Ensure it is created with appropriate Spark SQL data types, including `w_insert_dt` and `w_update_dt` as `TIMESTAMP`, and `USING DELTA` clause.
4.  **ETL_PROC_WID**: The `ETL_PROC_WID` in the original script was a hardcoded `12345`. It has been converted to use a Databricks widget `${ETL_PROC_WID}` for dynamic assignment. Ensure this widget is set appropriately before execution.
5.  **Date Filtering**: The incremental filtering `b.LAST_UPDATE_DATE > TO_DATE('2024-01-01', 'YYYY-MM-DD') AND b.LAST_UPDATE_DATE <= SYSDATE` uses a hardcoded start date `'2024-01-01'`. Consider replacing this with a dynamic parameter (e.g., from an ETL control table or another widget) if historical loads need to be managed flexibly.
6.  **`current_timestamp()`**: `SYSDATE` was converted to `current_timestamp()` to maintain time component consistency with `LAST_UPDATE_DATE` which is `TIMESTAMP`.